In [47]:
from tensorflow import keras
from tensorflow.keras import layers


In [48]:
import yfinance as yf
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

In [49]:
stock = yf.Ticker("^NSEI") 
end_date = pd.Timestamp.now()
start_date = end_date - pd.DateOffset(years=5)
data = stock.history(start=start_date, end=end_date)

In [50]:
data.head()

,Open,High,Low,Close,Volume,Dividends,Stock Splits
Date,,,,,,,
2020-04-07 00:00:00+05:30,8446.299805,8819.400391,8360.950195,8792.200195,814200,0.0,0.0
2020-04-08 00:00:00+05:30,8688.900391,9131.700195,8653.900391,8748.750000,896500,0.0,0.0
2020-04-09 00:00:00+05:30,8973.049805,9128.349609,8904.549805,9111.900391,742100,0.0,0.0
2020-04-13 00:00:00+05:30,9103.950195,9112.049805,8912.400391,8993.849609,644000,0.0,0.0
2020-04-15 00:00:00+05:30,9196.400391,9261.200195,8874.099609,8925.299805,879100,0.0,0.0


In [51]:
def preprocess_data(df):
    # Handle missing values
    df = df.ffill().bfill()    
    # Handle outliers using IQR method
    def remove_outliers(series):
        Q1 = series.quantile(0.25)
        Q3 = series.quantile(0.75)
        IQR = Q3 - Q1
        lower_bound = Q1 - 1.5 * IQR
        upper_bound = Q3 + 1.5 * IQR
        return series[(series >= lower_bound) & (series <= upper_bound)]
    
    # Remove price and volume outliers
    df['Close'] = remove_outliers(df['Close'])
    df['Volume'] = remove_outliers(df['Volume'])
    
    # Calculate technical indicators
    # 1. Moving Averages
    df['SMA_20'] = df['Close'].rolling(window=20).mean()
    df['SMA_50'] = df['Close'].rolling(window=50).mean()
    
    # 2. RSI
    delta = df['Close'].diff()
    gain = (delta.where(delta > 0, 0)).rolling(window=14).mean()
    loss = (-delta.where(delta < 0, 0)).rolling(window=14).mean()
    rs = gain / loss
    df['RSI'] = 100 - (100 / (1 + rs))
    
    # 3. MACD
    exp1 = df['Close'].ewm(span=12, adjust=False).mean()
    exp2 = df['Close'].ewm(span=26, adjust=False).mean()
    df['MACD'] = exp1 - exp2
    df['Signal_Line'] = df['MACD'].ewm(span=9, adjust=False).mean()
    
    # Calculate returns
    df['Daily_Return'] = df['Close'].pct_change()
    
    return df

# Apply preprocessing
data = preprocess_data(data)

In [52]:
def prepare_for_ml(data):

    data['Target'] = (data['Close'].shift(-1) > data['Close']).astype(int)
    features = [
        'SMA_20', 'SMA_50', 'RSI', 'MACD', 'Signal_Line', 
        'Daily_Return'
    ]
    
    data = data.dropna()
    

    X = data[features]
    y = data['Target']
    
    return X, y

In [53]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from tensorflow import keras
from tensorflow.keras import layers

In [54]:
X,Y = prepare_for_ml(data)
X_train, X_test, y_train, y_test = train_test_split(X, Y, test_size=0.2, random_state=42)
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

In [55]:
X_train.shape
input_shape = [6]

In [ ]:
model = keras.Sequential([
    layers.Dense(units = 512, activation='relu', input_shape=input_shape),
    layers.Dropout(0.3),
    layers.Dense(units = 256, activation='relu'),
    layers.Dropout(0.3),
    layers.Dense(units = 128, activation='relu'),
    layers.Dropout(0.3),
    layers.Dense(units = 64, activation='relu'),
    layers.Dense(units = 32, activation='relu'),
    layers.Dense(units = 16, activation='relu'),
    layers.Dense(units = 1, activation='sigmoid'),
]) 
optimizer = keras.optimizers.Adam(learning_rate=0.01)
model.compile(optimizer=optimizer, loss='binary_crossentropy', metrics=['accuracy'])
history = model.fit(X_train, y_train, epochs=50, batch_size=32, validation_split=0.2)

Epoch 1/100
24/24 ━━━━━━━━━━━━━━━━━━━━ 2s 11ms/step - accuracy: 0.5048 - loss: 0.7121 - val_accuracy: 0.5806 - val_loss: 0.6877
Epoch 2/100
24/24 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.5385 - loss: 0.6938 - val_accuracy: 0.5806 - val_loss: 0.6797
Epoch 3/100
24/24 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.5780 - loss: 0.6797 - val_accuracy: 0.5806 - val_loss: 0.6799
Epoch 4/100
24/24 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.5517 - loss: 0.6877 - val_accuracy: 0.5860 - val_loss: 0.6912
Epoch 5/100
24/24 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.5301 - loss: 0.6915 - val_accuracy: 0.5806 - val_loss: 0.6818
Epoch 6/100
24/24 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.5890 - loss: 0.6798 - val_accuracy: 0.5806 - val_loss: 0.6841
Epoch 7/100
24/24 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.5602 - loss: 0.6850 - val_accuracy: 0.5806 - val_loss: 0.6818
Epoch 8/100
24/24 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.5248 - loss: 0.6891 - val_accuracy: 0.5806 - 

In [44]:
model.summary()

Model: "sequential_4"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense_42 (Dense)                │ (None, 512)            │         3,584 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_43 (Dense)                │ (None, 256)            │       131,328 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_44 (Dense)                │ (None, 128)            │        32,896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_45 (Dense)                │ (None, 64)             │         8,256 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_46 (Dense)                │ (None, 32)             │         2,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_47 (Dense)                │ (None, 16)             │           528 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_48 (Dense)                │ (None, 1)              │            17 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 536,069 (2.04 MB)

 Trainable params: 178,689 (698.00 KB)

 Non-trainable params: 0 (0.00 B)

 Optimizer params: 357,380 (1.36 MB)

In [45]:
test_loss, test_accuracy = model.evaluate(X_test, y_test)
print(f"Test Accuracy: {test_accuracy}")



8/8 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.5523 - loss: 0.7178 
Test Accuracy: 0.5215517282485962
